In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from utils.prompts import get_system_prompt

In [2]:
%load_ext autoreload
%autoreload 2
import tiktoken
import glob
import pandas as pd
import json

In [3]:
DATA_DIR = "../dataset/safe-role/"
files = glob.glob(f"{DATA_DIR}*_dataset.csv")

df_list = [pd.read_csv(f) for f in files if "empty" not in f and "refused" not in f]
merged_df = pd.concat(df_list, axis=0, ignore_index=True)
test_df = merged_df[merged_df["split"] == "test"]

In [4]:
with open('../dataset/personas_desc.json', 'r') as f:
        personas = json.load(f)

In [5]:
prompts = []
for persona_name, persona_desc in personas.items():

    persona_instruction = get_system_prompt(persona_name, persona_desc)
    prompts.extend([persona_instruction + q
                for q in test_df[test_df['persona'] == persona_name]["prompt"]
            ])

In [6]:
enc = tiktoken.encoding_for_model("gpt-5")
tokens = enc.encode_batch(prompts)
input_tokens = sum([len(t) for t in tokens])
output_tokens = 256 * len(prompts)

In [7]:
# gpt-5.6-terra:
1.25*input_tokens/1e6 + 1.5625*output_tokens/1e6

1.8997312499999999

In [8]:
# gpt-5.6-sol:
2.5*input_tokens/1e6 + 3.125*output_tokens/1e6

3.7994624999999997

In [9]:
# gpt-5.6-luna:
0.5*input_tokens/1e6 + 0.625*output_tokens/1e6

0.7598925000000001